# 10과 - 프로덕션 환경에서의 AI 에이전트

이 과제에서는 Microsoft Agent Framework (Python)를 사용한 AI 에이전트의 <strong>프로덕션 패턴</strong>을 배웁니다. 다음을 다룹니다:

- **관찰 가능성** — 에이전트 상호작용에 타이밍과 로깅 추가
- <strong>평가</strong> — 평가자 에이전트를 사용하여 응답 품질 점수화
- **비용 관리** — 토큰 최적화 및 모델 선택 전략

시나리오는 사용자 여행 계획을 돕는 <strong>여행 에이전트</strong>이며, 그 위에 모니터링과 평가가 추가됩니다.


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 프로덕션 고려사항

AI 에이전트를 프로토타입에서 프로덕션으로 이동하려면 세 가지 핵심 요소에 주의해야 합니다:

1. **관찰 가능성** — 에이전트가 무엇을 하고 있는지, 얼마나 걸리는지, 어떤 도구를 호출하는지에 대한 가시성이 필요합니다. 추적 및 로깅 없이는 프로덕션 문제를 디버깅하는 것이 거의 불가능합니다.

2. <strong>평가</strong> — 자동화된 품질 검사를 통해 에이전트의 응답이 시간이 지나도 정확하고 완전하며 도움이 되는지 보장합니다. 평가자 에이전트는 정의된 기준에 따라 응답을 점수화할 수 있습니다.

3. **비용 관리** — 토큰 사용량은 비용에 직접적인 영향을 미칩니다. 프롬프트 최적화, 모델 선택, 캐싱과 같은 전략은 품질을 희생하지 않고 비용을 통제하는 데 도움이 됩니다.


## 관찰 가능한 에이전트 구축

우리는 여행 도구를 정의하고 에이전트 호출을 타이밍으로 감싸서 대기 시간을 모니터링할 수 있습니다. 실제 환경에서는 OpenTelemetry 또는 유사한 추적 백엔드와 통합할 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 비용 관리 전략

비용 관리는 생산 AI 에이전트에 매우 중요합니다. 주요 전략은 다음과 같습니다:

| 전략 | 설명 |
|---|---|
| **프롬프트 최적화** | 시스템 지침을 간결하게 유지하세요. 입력 토큰을 줄이기 위해 중복된 컨텍스트를 제거하세요. |
    "| **모델 선택** | 분류나 추출과 같은 간단한 작업엔 더 작고 저렴한 모델(e.g. GPT-5-mini)을 사용하고, 복잡한 추론에는 더 큰 모델을 예약하세요. |\n",
| <strong>캐싱</strong> | 중복된 API 호출을 방지하기 위해 도구 결과와 자주 하는 쿼리를 캐시하세요. |
| **토큰 한도 설정** | 예기치 않게 긴 응답을 방지하기 위해 `max_tokens` 제한을 설정하세요. |
| <strong>배칭</strong> | 가능한 경우 여러 사용자 쿼리를 하나의 API 호출로 그룹화하세요. |

실제로는 단계적 접근법이 효과적입니다: 간단한 요청은 빠르고 저렴한 모델로 처리하고, 복잡한 쿼리만 더 강력한 모델로 확장하세요.


## 요약

이 강의에서 배운 내용은 다음과 같습니다:

1. **관찰 가능성 추가**: 타이밍과 로깅을 통해 에이전트 상호작용에 관찰 가능성을 추가하여 추적 및 모니터링의 기초를 마련합니다.
2. **에이전트 응답 평가**: 완성도, 정확성, 유용성을 점수화하는 평가 에이전트를 사용하여 자동으로 응답을 평가합니다.
3. **비용 관리**: 프롬프트 최적화, 모델 선택, 캐싱, 토큰 예산을 통해 비용을 관리합니다.

이러한 생산 패턴은 AI 에이전트가 대규모에서 신뢰할 수 있고, 측정 가능하며, 비용 효율적임을 보장하는 데 도움이 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
